In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/raw/spotify-tracks-dataset.csv")
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"])
df = df.dropna(subset=["track_name"])

features = ["danceability", "energy", "valence", "acousticness", "speechiness", "instrumentalness", "tempo"]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[features])

df.shape

(113999, 20)

In [4]:
from sklearn.neighbors import NearestNeighbors

nn_model = NearestNeighbors(n_neighbors=11, metric="cosine")
nn_model.fit(scaled_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [5]:
def get_recommendations(track_index, n=10):
    distances, indices = nn_model.kneighbors([scaled_features[track_index]])
    
    similar_indices = indices[0][1:]
    similar_distances = distances[0][1:]
    
    recommendations = df.iloc[similar_indices][["track_name", "artists", "track_genre"]].copy()
    recommendations["similarity"] = 1 - similar_distances
    
    return recommendations

In [6]:
sample_track = df.iloc[0]
print(sample_track["track_name"], "-", sample_track["artists"])

get_recommendations(0)

Comedy - Gen Hoshino


,track_name,artists,track_genre,similarity
99152,Comedy,Gen Hoshino,singer-songwriter,1.000000
102151,Comedy,Gen Hoshino,songwriter,1.000000
62102,Comedy,Gen Hoshino,j-pop,1.000000
28888,Mary Long Tongue,Half Pint,dub,0.984216
850,Pop Virus,Gen Hoshino,acoustic,0.977689
21975,She's Royal,Tarrus Riley,dancehall,0.970972
28988,Night Nurse,Gregory Isaacs,dub,0.963773
66890,The Walmart Shuffle,Cupid,kids,0.963746
28847,Give It Up,Horace Andy,dub,0.963169
63595,So Nice,Natural Vibrations,j-rock,0.962766


In [7]:
def get_recommendations_v2(track_index, n=10):
    target_genre = df.iloc[track_index]["track_genre"]
    
    same_genre_mask = df["track_genre"] == target_genre
    same_genre_indices = df[same_genre_mask].index.to_numpy()
    
    genre_features = scaled_features[same_genre_indices]
    
    nn_genre = NearestNeighbors(n_neighbors=min(n + 1, len(same_genre_indices)), metric="cosine")
    nn_genre.fit(genre_features)
    
    target_position = np.where(same_genre_indices == track_index)[0][0]
    distances, local_indices = nn_genre.kneighbors([scaled_features[track_index]])
    
    global_indices = same_genre_indices[local_indices[0]]
    
    mask = global_indices != track_index
    global_indices = global_indices[mask][:n]
    result_distances = distances[0][mask][:n]
    
    recommendations = df.loc[global_indices][["track_name", "artists", "track_genre"]].copy()
    recommendations["similarity"] = 1 - result_distances
    
    return recommendations

In [9]:
import numpy as np
get_recommendations_v2(0)

,track_name,artists,track_genre,similarity
850,Pop Virus,Gen Hoshino,acoustic,0.977689
357,Look For The Good (Single Version),Jason Mraz,acoustic,0.948026
5,Days I Will Remember,Tyrone Wells,acoustic,0.908341
388,接吻,Hanare Gumi,acoustic,0.835247
488,Blister In The Sun,Violent Femmes,acoustic,0.833397
837,Living in the Moment,Jason Mraz,acoustic,0.785756
936,I Am So Mad at You,AJJ,acoustic,0.784521
751,FUSHIGI,Gen Hoshino,acoustic,0.784194
249,Christmas Comes But Once A Year,Albert King,acoustic,0.778749
248,Christmas Comes But Once A Year,Albert King,acoustic,0.778749
